# Fine-tuning QLoRA — Q&A médical (Mac / Apple Silicon)

Version adaptée du notebook Colab pour tourner **en local sur un Mac Apple Silicon**, avec **MLX** (le framework Metal d'Apple).

### Pourquoi une réécriture et pas un simple portage ?
Le notebook d'origine repose sur **Unsloth + bitsandbytes** (`load_in_4bit`, `adamw_8bit`), qui sont **strictement CUDA / NVIDIA** — ça ne tourne pas sur Mac, point. Sur Apple Silicon, le bon outil pour faire du LoRA quantisé (= QLoRA), c'est **MLX** : 4-bit natif, rapide, conçu par Apple pour Metal.

| Colab (origine) | Mac (cette version) |
|---|---|
| `unsloth` / `FastLanguageModel` | `mlx-lm` (`mlx_lm.lora`) |
| `load_in_4bit` (bitsandbytes) | modèle `mlx-community/...-4bit` (QLoRA natif) |
| `SFTTrainer` + `train_on_responses_only` | `mlx_lm.lora --mask-prompt` |
| `adamw_8bit` | optimiseur MLX (AdamW) |
| Google Drive (éphémère) | disque local (rien à monter) |
| `save_pretrained_gguf` (unsloth) | `mlx_lm.fuse` (+ llama.cpp pour GGUF) |

### Prérequis
Mac **Apple Silicon** (M1/M2/M3/M4), macOS ≥ 13.5, Python ≥ 3.9. Lance ce notebook dans un **venv dédié**.

### RAM (mémoire unifiée) → taille de modèle conseillée
- **8 Go** : 1–3B uniquement (`BATCH_SIZE=1`, `MAX_SEQ_LEN=512`)
- **16 Go** : 7B 4-bit OK (`BATCH_SIZE` 1–2, seq 1024), 3B confortable
- **24–32 Go** : 7–9B confortable
- **36 Go+** : 7–13B sans souci

Ordre des cellules : install → check env → config → données → config YAML → train → test → fuse/Ollama (optionnel).

In [8]:
import platform, sys
is_mac = platform.system() == "Darwin"
is_arm = platform.machine() == "arm64"
print(f"macOS={is_mac} | Apple Silicon (arm64)={is_arm} | Python={sys.version.split()[0]}")
assert is_mac and is_arm, (
    "MLX nécessite un Mac Apple Silicon (M1/M2/M3/M4). "
    "Sur un Mac Intel, MLX ne fonctionne pas -> GPU NVIDIA (Colab) ou CPU PyTorch (lent)."
)
import mlx.core as mx, mlx_lm
print("MLX device:", mx.default_device(), "| mlx-lm", getattr(mlx_lm, "__version__", "?"))

macOS=True | Apple Silicon (arm64)=True | Python=3.13.3
MLX device: Device(gpu, 0) | mlx-lm 0.31.3


In [9]:
import os

# ---------------------------- CONFIG ----------------------------
CSV_PATH    = "../datasets/medical/dialogues_clean 2.csv"   # <-- ton CSV en local (même dossier que le notebook)
DATA_DIR    = "./data"                  # MLX y lira train.jsonl / valid.jsonl / test.jsonl
ADAPTER_DIR = "./adapters"              # adaptateurs LoRA sauvegardés ici
CONFIG_YAML = "lora_config.yaml"

MAX_SEQ_LEN   = 1024
MAX_SAMPLES   = 5000     # 1ère run : 3000-5000 suffit. Monte ensuite (le Mac est plus lent qu'un A100).
MIN_ANSWER_CH = 60       # on jette les réponses Doctor plus courtes
SEED          = 3407

# Repli l'instruction dans le tour user (Mistral/Gemma n'ont pas de rôle system). "" = rien.
SYSTEM_PROMPT = (
    "You are a knowledgeable and responsible medical doctor. "
    "Answer the patient's question clearly, and recommend seeing a clinician "
    "in person when appropriate."
)

# Modèles MLX pré-quantisés 4-bit (équivalents des bnb-4bit du notebook Colab).
# Tous validés présents sur le Hub. mistral/qwen/phi = ouverts ; gemma/llama = licence à accepter (token HF).
MODELS = {
    "mistral": "mlx-community/Mistral-7B-Instruct-v0.3-4bit",
    "qwen":    "mlx-community/Qwen2.5-7B-Instruct-4bit",     # plus léger : "mlx-community/Qwen2.5-3B-Instruct-4bit"
    "phi":     "mlx-community/Phi-3.5-mini-instruct-4bit",   # ~3.8B, idéal 8-16 Go
    "gemma":   "mlx-community/gemma-2-9b-it-4bit",           # 9B -> 24 Go+ conseillé (licence Gemma)
    "llama3b": "mlx-community/Llama-3.2-3B-Instruct-4bit",   # petit, rapide (licence Llama)
}
MODEL_ID = MODELS["phi"]   # <-- change la clé : "qwen" / "phi" / "gemma" / "llama3b"

# Hyperparamètres LoRA / entraînement
NUM_LAYERS    = 16      # nb de couches fine-tunées depuis le haut (-1 = toutes, plus lent/plus complet)
BATCH_SIZE    = 2       # 16 Go -> 1 ou 2 ; 8 Go -> 1
LEARNING_RATE = 2e-4    # si la loss diverge, redescends à 1e-4
EPOCHS        = 1       # sert à calculer --iters d'après la taille du train set
LORA_RANK     = 16      # r dans le notebook Colab
LORA_SCALE    = 20.0    # équivalent MLX de lora_alpha (MLX applique 'scale' directement)
LORA_DROPOUT  = 0.0
EXPORT_GGUF   = False    # True = tenter un export GGUF pour Ollama (voir dernière cellule)

os.makedirs(DATA_DIR, exist_ok=True)
print("Modèle choisi :", MODEL_ID)

Modèle choisi : mlx-community/Phi-3.5-mini-instruct-4bit


In [10]:
import re, json, random
import pandas as pd

# --- nettoyage (identique au notebook Colab : openers/closers boilerplate, réponses tronquées, artefacts `-->`) ---
_GREETING = re.compile(
    r"^\s*(hi|hello|hey|dear|welcome|thanks?|thank you|good (morning|afternoon|evening))\b[^.!?\n]*[.!?]\s*",
    re.IGNORECASE,
)
_CLOSERS = re.compile(
    r"\s*(hope (this|i)\s+(help|answered)[^.!?\n]*[.!?]|take care[^.!?\n]*[.!?]|(kind |warm )?regards.*|for (further|more) (info|information|queries?)[^.!?\n]*[.!?]?)\s*$",
    re.IGNORECASE | re.DOTALL,
)

def clean(txt):
    if not isinstance(txt, str):
        return ""
    txt = txt.split("-->")[0]               # vire les artefacts de liens strippés
    txt = re.sub(r"\s+", " ", txt).strip()
    txt = _GREETING.sub("", txt)            # vire l'ouverture "Hi doctor, ..."
    txt = _CLOSERS.sub("", txt).strip()     # vire la formule de politesse finale
    return txt

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip().lower() for c in df.columns]
assert {"patient", "doctor"}.issubset(df.columns), f"Colonnes trouvées : {list(df.columns)}"
df["patient"] = df["patient"].map(clean)
df["doctor"]  = df["doctor"].map(clean)
df = df[df["patient"].str.len() > 10]
df = df[df["doctor"].str.len() >= MIN_ANSWER_CH]
df = df.drop_duplicates(subset=["patient", "doctor"]).reset_index(drop=True)
print(f"Exemples après nettoyage : {len(df):,}")
if MAX_SAMPLES:
    df = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)

_SYS = (SYSTEM_PROMPT + "\n\n") if SYSTEM_PROMPT else ""

def to_record(row):
    # Format 'chat' MLX. Le system est replié dans le user (compatible Mistral/Gemma/Qwen/Phi).
    # Avec --mask-prompt, la loss ne porte QUE sur le dernier message (la réponse assistant).
    return {"messages": [
        {"role": "user", "content": _SYS + row["patient"]},
        {"role": "assistant", "content": row["doctor"]},
    ]}

records = [to_record(r) for _, r in df.iterrows()]
random.Random(SEED).shuffle(records)

n = len(records)
n_val  = max(1, int(n * 0.02))
n_test = max(1, int(n * 0.02))
valid = records[:n_val]
test  = records[n_val:n_val + n_test]
train = records[n_val + n_test:]

def dump_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

dump_jsonl(os.path.join(DATA_DIR, "train.jsonl"), train)
dump_jsonl(os.path.join(DATA_DIR, "valid.jsonl"), valid)
dump_jsonl(os.path.join(DATA_DIR, "test.jsonl"),  test)
print(f"train={len(train):,} | valid={len(valid):,} | test={len(test):,}  ->  {DATA_DIR}")
print("Exemple :", json.dumps(train[0], ensure_ascii=False)[:300])

Exemples après nettoyage : 237,563
train=4,800 | valid=100 | test=100  ->  ./data
Exemple : {"messages": [{"role": "user", "content": "You are a knowledgeable and responsible medical doctor. Answer the patient's question clearly, and recommend seeing a clinician in person when appropriate.\n\nI was just having a normal day and then people were walking by me so I sucked in my stomach and th


In [11]:
import yaml, math

# --iters = nb de pas d'entraînement. Ici ~EPOCHS passages sur le train set.
ITERS = max(100, math.ceil(len(train) / BATCH_SIZE) * EPOCHS)

cfg = {
    "model": MODEL_ID,
    "train": True,
    "data": DATA_DIR,
    "adapter_path": ADAPTER_DIR,
    "fine_tune_type": "lora",
    "num_layers": NUM_LAYERS,
    "batch_size": BATCH_SIZE,
    "iters": ITERS,
    "learning_rate": LEARNING_RATE,
    "max_seq_length": MAX_SEQ_LEN,
    "steps_per_report": 10,
    "steps_per_eval": 200,
    "save_every": 200,
    "val_batches": 25,
    "seed": SEED,
    "lora_parameters": {
        "rank": LORA_RANK,
        "scale": LORA_SCALE,
        "dropout": LORA_DROPOUT,
    },
}
with open(CONFIG_YAML, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print(f"--iters calculé : {ITERS}\n")
print(open(CONFIG_YAML).read())

--iters calculé : 2400

model: mlx-community/Phi-3.5-mini-instruct-4bit
train: true
data: ./data
adapter_path: ./adapters
fine_tune_type: lora
num_layers: 16
batch_size: 2
iters: 2400
learning_rate: 0.0002
max_seq_length: 1024
steps_per_report: 10
steps_per_eval: 200
save_every: 200
val_batches: 25
seed: 3407
lora_parameters:
  rank: 16
  scale: 20.0
  dropout: 0.0



In [ ]:
# Entraînement QLoRA. --mask-prompt = on n'entraîne QUE sur la réponse du médecin
# (équivalent de train_on_responses_only côté Unsloth). Le checkpoint est repris automatiquement
# si tu relances et qu'un adaptateur existe déjà dans ADAPTER_DIR.
!python -m mlx_lm.lora --config {CONFIG_YAML} --mask-prompt

Calling `python -m mlx_lm.lora...` directly is deprecated. Use `mlx_lm.lora...` or `python -m mlx_lm lora ...` instead.
Loading configuration file lora_config.yaml
Loading pretrained model
Fetching 11 files: 100%|███████████████████████| 11/11 [03:24<00:00, 18.56s/it]
Download complete: 100%|██████████████████| 2.15G/2.15G [03:24<00:00, 10.5MB/s]
This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
Loading datasets
Training
Trainable parameters: 0.329% (12.583M/3821.080M)
Starting training..., iters: 2400
Calculating loss...: 100%|█████████████████████| 25/25 [00:30<00:00,  1.21s/it]
Iter 1: Val loss 4.234, Val took 30.329s
Iter 10: Train loss 2.996, Learning Rate 2.000

In [ ]:
from mlx_lm import load, generate

# Recharge le modèle de base + les adaptateurs LoRA fraîchement entraînés.
model, tok = load(MODEL_ID, adapter_path=ADAPTER_DIR)

q = "I have had a mild headache and a slight fever for two days. What should I do?"
msgs = [{"role": "user", "content": _SYS + q}]   # même mise en forme qu'à l'entraînement
prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

resp = generate(model, tok, prompt=prompt, max_tokens=256, verbose=True)

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 2232.20it/s]


RuntimeError: [load_safetensors] Failed to open file adapters/adapters.safetensors

### Optionnel — fusionner le LoRA et exporter

`mlx_lm.fuse` fusionne les adaptateurs dans le modèle de base et produit un modèle autonome (safetensors).

**Pas besoin d'Ollama pour tourner sur Mac** : tu peux servir le modèle fusionné directement avec MLX
```bash
python -m mlx_lm.generate --model ./fused-model --prompt "..."
python -m mlx_lm.server   --model ./fused-model     # API type OpenAI en local (http://localhost:8080)
```

**Pour Ollama (GGUF) :** `--export-gguf` (sur `mlx_lm.fuse`) ne marche que pour **Llama / Mistral / Mixtral**. Pour Qwen / Phi / Gemma, fusionne en safetensors (cellule ci-dessous) puis convertis avec `llama.cpp` (`convert_hf_to_gguf.py`) avant `ollama create`. Côté Ollama, crée un `Modelfile` (`FROM ./xxx.gguf`) puis `ollama create medical-doc -f Modelfile`.

In [ ]:
FUSED_DIR = "./fused-model"
fuse_cmd = f"python -m mlx_lm.fuse --model {MODEL_ID} --adapter-path {ADAPTER_DIR} --save-path {FUSED_DIR}"

gguf_ok = any(k in MODEL_ID.lower() for k in ["mistral", "llama", "mixtral"])
if EXPORT_GGUF and gguf_ok:
    fuse_cmd += " --export-gguf"

!{fuse_cmd}

print("Modèle fusionné (safetensors) ->", FUSED_DIR)
if EXPORT_GGUF and not gguf_ok:
    print("GGUF auto non supporté pour cette archi (Qwen/Phi/Gemma) : "
          "fusionne en safetensors puis convertis avec llama.cpp (convert_hf_to_gguf.py).")